# 🛡️ Aegis — De-identification Pipeline

This notebook walks you through the complete Aegis pipeline for de-identifying medical images (DICOM series, individual DICOMs, JPEG, PNG).

**Three Pipeline Modes:**
- **DICOM single-file** — `LoadDicomRawd → RedactPixelPHId → ScrubDicomMetadatad → SaveDicomd`
- **DICOM series** — `LoadDicomSeriesd → RedactPixelPHId (keyframe OCR) → ScrubDicomMetadatad (geometry-preserving) → SaveDicomSeriesd`
- **Image (JPEG/PNG)** — `LoadImaged → RedactPixelPHId → SaveImaged`

`RedactPixelPHId` is **shared** across all three pipelines — it operates on pixel arrays regardless of format.

**MONAI API Compliance:** All transforms inherit from `MapTransform`, `InvertibleTransform`, or `ThreadUnsafe` as appropriate.

---
## 1. Setup & Imports

In [ ]:
---
## 2. Load Configuration

The pipeline is driven by `config.yaml`, which controls OCR, NER, clinical allowlists, PHI heuristics, PII tag actions, series processing, storage backend, and tokenization.

In [ ]:
from config.config_loader import load_config

CONFIG_PATH = 'monai_aegis/config/config.yaml'

config = load_config(CONFIG_PATH)

print('=== Paths ===')
for k, v in config.get('paths', {}).items():
    print(f'  {k}: {v}')

print(f'\n=== Storage ===')
for k, v in config.get('storage', {}).items():
    print(f'  {k}: {v}')

print(f'\n=== Tokenization ===')
for k, v in config.get('tokenization', {}).items():
    print(f'  {k}: {v or "(default)"}')

print(f'\n=== Series Settings ===')
for k, v in config.get('series', {}).items():
    print(f'  {k}: {v}')

print(f'\n=== OCR Settings ===')
for k, v in config['ocr'].items():
    print(f'  {k}: {v}')

print(f'\n=== NER Settings ===')
print(f'  enabled: {config["ner"]["enabled"]}')
print(f'  model:   {config["ner"]["model_name"]}')
print(f'  device:  {config["ner"]["device"]}')
print(f'  PHI labels ({len(config["ner"]["phi_labels"])}): {config["ner"]["phi_labels"][:5]}...')

print(f'\n=== PII Mapping ===')
for tag, action in config['pii_mapping'].items():
    print(f'  {tag} → {action}')

---
## 2. Load Configuration

The pipeline is driven by `config.yaml`, which controls OCR settings, NER model, clinical allowlists, PHI heuristics, PII tag actions, and series processing options.

In [ ]:
import yaml

CONFIG_PATH = 'monai_aegis/config/config.yaml'

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

print('=== Paths ===')
for k, v in config.get('paths', {}).items():
    print(f'  {k}: {v}')

print(f'\n=== Series Settings ===')
for k, v in config.get('series', {}).items():
    print(f'  {k}: {v}')

print(f'\n=== OCR Settings ===')
for k, v in config['ocr'].items():
    print(f'  {k}: {v}')

print(f'\n=== NER Settings ===')
print(f'  enabled: {config["ner"]["enabled"]}')
print(f'  model:   {config["ner"]["model_name"]}')
print(f'  device:  {config["ner"]["device"]}')
print(f'  PHI labels ({len(config["ner"]["phi_labels"])}): {config["ner"]["phi_labels"][:5]}...')

print(f'\n=== PII Mapping ===')
for tag, action in config['pii_mapping'].items():
    print(f'  {tag} → {action}')

---
## 3. Inspect Input Files

Let's see what files are available in `staging_input/` (including subdirectories).

In [ ]:
INPUT_DIR = 'staging_input'
OUTPUT_DIR = 'staging_output'

# Recursively scan for all supported files
all_files = []
for root, _dirs, fnames in os.walk(INPUT_DIR):
    for f in fnames:
        if f.lower().endswith(('.dcm', '.jpg', '.jpeg', '.png')):
            all_files.append(os.path.relpath(os.path.join(root, f), INPUT_DIR))
all_files.sort()

dcm_files = [f for f in all_files if f.lower().endswith('.dcm')]
img_files = [f for f in all_files if not f.lower().endswith('.dcm')]

print(f'Found {len(all_files)} input files ({len(dcm_files)} DICOM, {len(img_files)} images)\n')
for f in all_files:
    full = os.path.join(INPUT_DIR, f)
    size_kb = os.path.getsize(full) / 1024
    ext = os.path.splitext(f)[1].upper()
    print(f'  {ext:6s}  {size_kb:7.1f} KB  {f}')

---
## 4. Preview an Input Image (Before De-identification)

View the original image with PHI visible.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import pydicom
import numpy as np

def show_image(filepath, title=''):
    """Display a DICOM or standard image file."""
    if filepath.lower().endswith('.dcm'):
        ds = pydicom.dcmread(filepath)
        img = ds.pixel_array
    else:
        img = np.array(Image.open(filepath))
    
    plt.figure(figsize=(10, 8))
    if img.ndim == 2:
        plt.imshow(img, cmap='gray')
    else:
        plt.imshow(img)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# ⬇️ Change this to any file from staging_input/
SAMPLE_FILE = '202601170918250004ABD.JPG'

show_image(os.path.join(INPUT_DIR, SAMPLE_FILE), 
           title=f'BEFORE — {SAMPLE_FILE} (PHI visible)')

---
## 5. DICOM Single-File Pipeline — Build & Run

Build the single-file pipeline and process a single file.

In [ ]:
from transforms.pipeline import build_pipeline

# Build the single-file pipeline (loads config, initializes all transforms)
pipeline = build_pipeline(
    config_path=os.path.abspath(CONFIG_PATH),
    output_dir=OUTPUT_DIR
)

print('✅ DICOM single-file pipeline built successfully')
print(f'   Steps: LoadDicomRawd → RedactPixelPHId → ScrubDicomMetadatad → SaveDicomd')

In [ ]:
# Process one file
sample_path = os.path.join(INPUT_DIR, SAMPLE_FILE)
result = pipeline({'image': sample_path})

print(f'✅ Processed: {SAMPLE_FILE}')
print(f'\nResult dictionary keys: {list(result.keys())}')

---
## 6. Inspect Redaction Statistics

The pipeline attaches redaction statistics to the result dictionary.

In [ ]:
stats = result.get('image_redaction_stats', {})

print('=== Redaction Statistics ===')
print(f'  Total text detections:   {stats.get("total_detections", 0)}')
print(f'  Redacted (PHI):          {stats.get("redacted_count", 0)}')
print(f'  Preserved (clinical):    {stats.get("preserved_count", 0)}')
print(f'  Low confidence (skipped): {stats.get("low_confidence_count", 0)}')

if stats.get('low_confidence_count', 0) > 0:
    print('\n⚠️  This image has low-confidence regions — should be flagged for manual review.')
else:
    print('\n✅  All text regions processed with confidence above threshold.')

---
## 7. View the De-identified Output (After)

Compare the original and de-identified images side by side.

In [ ]:
# For standard images, extract from the result tensor
img_tensor = result['image']

if hasattr(img_tensor, 'cpu'):
    img_array = img_tensor.cpu().numpy()
else:
    img_array = np.array(img_tensor)

# Convert channel-first (C, H, W) → (H, W, C)
if img_array.ndim == 3:
    if img_array.shape[0] == 1:
        img_array = img_array.squeeze(0)
    elif img_array.shape[0] == 3:
        img_array = np.moveaxis(img_array, 0, -1)

# Normalize float to uint8
if img_array.dtype in [np.float32, np.float64]:
    if img_array.max() <= 1.1:
        img_array = (img_array * 255).astype(np.uint8)
    else:
        img_array = img_array.astype(np.uint8)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

original = np.array(Image.open(sample_path))
axes[0].imshow(original)
axes[0].set_title(f'BEFORE — {SAMPLE_FILE}', fontsize=13, color='red')
axes[0].axis('off')

if img_array.ndim == 2:
    axes[1].imshow(img_array, cmap='gray')
else:
    axes[1].imshow(img_array)
axes[1].set_title(f'AFTER — PHI Redacted', fontsize=13, color='green')
axes[1].axis('off')

plt.suptitle('Aegis De-identification', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Series-Aware Pipeline — Discover & Process DICOM Series

Discover DICOM files, group into series, validate geometry, sort slices, and process as volumes.

In [ ]:
from transforms.discovery import discover_dicoms, group_into_series, validate_series, sort_slices

# Step 1: Discover all DICOMs recursively
slices = discover_dicoms(INPUT_DIR)
print(f'Discovered {len(slices)} DICOM slices\n')

# Step 2: Group by (StudyUID, SeriesUID)
series_groups = group_into_series(slices)
print(f'Grouped into {len(series_groups)} series:\n')
for (study_uid, series_uid), group in series_groups.items():
    print(f'  Study: {study_uid[:20]}...')
    print(f'  Series: {series_uid[:20]}...')
    print(f'  Slices: {len(group)}')
    print(f'  Modality: {group[0].modality}')
    print(f'  Size: {group[0].rows} × {group[0].columns}')
    print()

In [ ]:
# Step 3: Validate geometry and sort
for (study_uid, series_uid), group in series_groups.items():
    sub_series = validate_series(group)
    print(f'Series {series_uid[:20]}... → {len(sub_series)} sub-series')
    
    for i, sub in enumerate(sub_series):
        sorted_sub = sort_slices(sub)
        print(f'  Sub-series {i}: {len(sorted_sub)} slices, sorted by:', end=' ')
        if all(s.image_position_patient is not None for s in sorted_sub):
            print('ImagePositionPatient (z-coordinate)')
        elif all(s.instance_number is not None for s in sorted_sub):
            print('InstanceNumber')
        else:
            print('filename')

In [ ]:
from transforms.pipeline import build_series_pipeline

# Build the series-aware pipeline
series_pipeline = build_series_pipeline(
    config_path=os.path.abspath(CONFIG_PATH),
    output_dir=OUTPUT_DIR,
    input_dir=INPUT_DIR
)

print('✅ Series pipeline built successfully')
print(f'   Steps: LoadDicomSeriesd → RedactPixelPHId (keyframe OCR) → ScrubDicomMetadatad → SaveDicomSeriesd')

In [ ]:
# Process each series
for (study_uid, series_uid), group in series_groups.items():
    sub_series_list = validate_series(group)
    
    for sub_idx, sub in enumerate(sub_series_list):
        sorted_sub = sort_slices(sub)
        filepaths = [s.filepath for s in sorted_sub]
        
        print(f'\nProcessing series {series_uid[:20]}... ({len(filepaths)} slices)')
        
        result = series_pipeline({'image': filepaths})
        
        stats = result.get('image_redaction_stats', {})
        print(f'  ✅ Strategy:   {stats.get("volume_strategy", "N/A")}')
        print(f'  Keyframes:     {stats.get("keyframe_indices", [])}')
        print(f'  Redacted:      {stats.get("redacted_count", 0)}')
        print(f'  Output shape:  {result["image"].shape}')

---
## 9. Batch Process All Files (CLI-equivalent)

Process every file in `staging_input/` — DICOMs as series, JPEGs/PNGs via the dedicated image pipeline.

In [ ]:
import shutil
from transforms.pipeline import build_image_pipeline

NOT_PROCESSED_DIR = 'staging_not_processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(NOT_PROCESSED_DIR, exist_ok=True)

processed, flagged, errors = 0, 0, 0

# --- Process DICOMs as series ---
slices = discover_dicoms(INPUT_DIR)
if slices:
    series_groups = group_into_series(slices)
    sp = build_series_pipeline(
        config_path=os.path.abspath(CONFIG_PATH),
        output_dir=OUTPUT_DIR,
        input_dir=INPUT_DIR
    )
    for (study_uid, series_uid), group in series_groups.items():
        for sub in validate_series(group):
            sorted_sub = sort_slices(sub)
            filepaths = [s.filepath for s in sorted_sub]
            try:
                sp({'image': filepaths})
                print(f'✅  Series {series_uid[:20]}... ({len(filepaths)} slices)')
                processed += len(filepaths)
            except Exception as e:
                print(f'❌  Series {series_uid[:20]}... — Error: {e}')
                errors += 1

# --- Process non-DICOM images via image pipeline ---
if img_files:
    img_pipeline = build_image_pipeline(
        config_path=os.path.abspath(CONFIG_PATH),
        output_dir=OUTPUT_DIR
    )
    for filepath in img_files:
        full_path = os.path.join(INPUT_DIR, filepath)
        try:
            result = img_pipeline({'image': full_path})
            stats = result.get('image_redaction_stats', {})
            low_conf = stats.get('low_confidence_count', 0)
            if low_conf > 0:
                np_dest = os.path.join(NOT_PROCESSED_DIR, filepath)
                os.makedirs(os.path.dirname(np_dest), exist_ok=True)
                shutil.copy2(full_path, np_dest)
                print(f'⚠️  {filepath} → staging_not_processed/ (low confidence)')
                flagged += 1
                continue
            print(f'✅  {filepath} → staging_output/')
            processed += 1
        except Exception as e:
            print(f'❌  {filepath} — Error: {e}')
            errors += 1

print(f'\n{"=" * 50}')
print(f'  Processed:      {processed}')
print(f'  Flagged:        {flagged}')
print(f'  Errors:         {errors}')
print(f'{'=' * 50}')

---
## 10. Inspect a DICOM File (Before & After Metadata Scrubbing)

Compare DICOM tags before and after scrubbing.

In [ ]:
DICOM_FILE = 'U0000001.dcm'  # ⬇️ Change to any .dcm in staging_input/

# Tags we care about (from pii_mapping in config)
pii_tags = {
    '(0010,0010)': 'PatientName',
    '(0010,0020)': 'PatientID',
    '(0010,0030)': 'PatientBirthDate',
    '(0008,0020)': 'StudyDate',
    '(0008,0050)': 'AccessionNumber',
    '(0008,0090)': 'ReferringPhysicianName',
    '(0008,0080)': 'InstitutionName',
}

def get_tag_value(ds, tag_str):
    """Safely get a DICOM tag value."""
    clean = tag_str.strip('() ').replace(' ', '')
    parts = clean.split(',')
    tag = pydicom.tag.Tag(int(parts[0], 16), int(parts[1], 16))
    if tag in ds:
        return str(ds[tag].value)
    return '<REMOVED>'

# Load original
ds_original = pydicom.dcmread(os.path.join(INPUT_DIR, DICOM_FILE))

# Load scrubbed (if available)
scrubbed_path = os.path.join(OUTPUT_DIR, DICOM_FILE)
if os.path.exists(scrubbed_path):
    ds_scrubbed = pydicom.dcmread(scrubbed_path)
    
    print(f'DICOM Tag Comparison: {DICOM_FILE}')
    print(f'{"Tag":<18} {"Field":<26} {"BEFORE":<30} {"AFTER"}')
    print('-' * 100)
    for tag, name in pii_tags.items():
        before = get_tag_value(ds_original, tag)
        after = get_tag_value(ds_scrubbed, tag)
        changed = '🔴' if before != after else '⚪'
        print(f'{changed} {tag:<16} {name:<26} {before:<30} {after}')
else:
    print(f'⚠️  Scrubbed DICOM not found at {scrubbed_path}')
    print(f'   Run the batch processing cell above first.')

---
## 11. Using Individual Transforms

You can use each transform independently for fine-grained control.

In [ ]:
from transforms.io import LoadDicomRawd
from transforms.pixel import RedactPixelPHId
from transforms.metadata import ScrubDicomMetadatad
from monai.transforms import MapTransform, InvertibleTransform

# Step 1: Load (Ingestion Zone)
loader = LoadDicomRawd(keys=['image'])
data = loader({'image': os.path.join(INPUT_DIR, SAMPLE_FILE)})

print(f'After LoadDicomRawd:')
print(f'  Tensor shape:       {data["image"].shape}')
print(f'  Tensor dtype:       {data["image"].dtype}')

# Enriched MetaTensor metadata
meta = data['image'].meta
print(f'\n=== Enriched MetaTensor.meta ===')
for k, v in meta.items():
    print(f'  {k}: {v}')

# Verify meta_dict is a live reference (not a copy)
is_ref = data['image_meta_dict'] is data['image'].meta
print(f'\n  meta_dict is live reference: {"✅ Yes" if is_ref else "❌ No (detached copy)"}')

if 'image_dicom_dataset' in data:
    print(f'  Cached dataset:     ✅ pydicom.Dataset in memory')
else:
    print(f'  Cached dataset:     — (non-DICOM file)')

In [ ]:
# Step 2: Redact (Logic Zone)
redactor = RedactPixelPHId(keys=['image'], config=config)
data = redactor(data)

stats = data.get('image_redaction_stats', {})
print(f'After RedactPixelPHId:')
print(f'  Total detections:  {stats.get("total_detections", 0)}')
print(f'  Redacted:          {stats.get("redacted_count", 0)}')
print(f'  Preserved:         {stats.get("preserved_count", 0)}')

---
## 12. Series Transform — Using LoadDicomSeriesd Directly

Load a DICOM series as a volume and inspect the `(C, D, H, W)` tensor.

In [ ]:
from transforms.series_io import LoadDicomSeriesd

# Use the first discovered series
if series_groups:
    key = list(series_groups.keys())[0]
    group = series_groups[key]
    sorted_group = sort_slices(group)
    filepaths = [s.filepath for s in sorted_group]
    
    loader = LoadDicomSeriesd(keys=['image'])
    data = loader({'image': filepaths})
    
    print(f'Loaded series: {key[1][:30]}...')
    print(f'  Volume shape: {data["image"].shape}  (C, D, H, W)')
    print(f'  Dtype:        {data["image"].dtype}')
    print(f'  Modality:     {data["image"].meta["modality"]}')
    print(f'  Num slices:   {data["image"].meta["num_slices"]}')
    print(f'  Multiframe:   {data["image"].meta["is_multiframe"]}')
    print(f'  Cached datasets: {len(data.get("image_dicom_datasets", []))}')
else:
    print('No DICOM series found in input directory.')

---
## 13. MONAI API Compliance Verification

Verify that all transforms inherit from the correct MONAI base classes.

In [ ]:
from transforms.io import LoadDicomRawd, SaveDicomd, LoadImaged, SaveImaged
from transforms.series_io import LoadDicomSeriesd, SaveDicomSeriesd
from transforms.pixel import RedactPixelPHId
from transforms.metadata import ScrubDicomMetadatad
from monai.transforms import MapTransform, InvertibleTransform, Transform, ThreadUnsafe

compliance = [
    ('LoadDicomRawd',         LoadDicomRawd,         [MapTransform]),
    ('LoadImaged',            LoadImaged,            [MapTransform]),
    ('LoadDicomSeriesd',      LoadDicomSeriesd,      [MapTransform]),
    ('RedactPixelPHId',       RedactPixelPHId,       [MapTransform, InvertibleTransform]),
    ('ScrubDicomMetadatad',   ScrubDicomMetadatad,   [MapTransform]),
    ('SaveDicomd',            SaveDicomd,            [MapTransform, ThreadUnsafe]),
    ('SaveImaged',            SaveImaged,            [MapTransform, ThreadUnsafe]),
    ('SaveDicomSeriesd',      SaveDicomSeriesd,      [MapTransform, ThreadUnsafe]),
]

print(f'{"Transform":<24} {"Expected Bases":<40} {"Status"}')
print('=' * 80)
for name, cls, bases in compliance:
    bases_str = ', '.join(b.__name__ for b in bases)
    status = '✅ PASS' if all(issubclass(cls, b) for b in bases) else '❌ FAIL'
    print(f'{name:<24} {bases_str:<40} {status}')

---
## 14. Run Unit Tests

Run the full test suite (97 tests) from within the notebook.

In [ ]:
!PYTHONPATH=monai_aegis python -m unittest discover tests/unit -v

---
## 15. Configuration Quick-Reference

| Setting | Location | Effect |
|---------|----------|--------|
| `paths.input_dir` | config.yaml | Root directory for input files |
| `paths.output_dir` | config.yaml | Root directory for de-identified output |
| `storage.protocol` | config.yaml | Storage backend: `file`, `s3`, `gs`, `az` |
| `storage.options` | config.yaml | fsspec kwargs (credentials, endpoint, etc.) |
| `tokenization.salt` | config.yaml / env var | Secret salt for deterministic tokenization |
| `series.enabled` | config.yaml | Enable series-aware processing |
| `series.keyframe_count` | config.yaml | Keyframes for volume OCR (default: 3) |
| `series.accepted_modalities` | config.yaml | DICOM modalities to process as series |
| `ocr.confidence_threshold` | config.yaml | Lower = more text detected; Higher = stricter |
| `ner.enabled` | config.yaml | `true` = Stanford NER; `false` = regex safelist |
| `ner.device` | config.yaml | `cpu`, `cuda`, or `mps` |
| `ner.phi_labels` | config.yaml | NER entity types treated as PHI |
| `ner.clinical_allowlist` | config.yaml | Terms that are NEVER redacted |
| `ner.clinical_patterns` | config.yaml | Regex for clinical text (preserved) |
| `ner.phi_heuristic_patterns` | config.yaml | Regex for PHI in fragments (redacted) |
| `pii_mapping` | config.yaml | DICOM tag actions: REMOVE / ZERO / DUMMY |